# 13 - Merge RegBL and Hydrogeo Watermass

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `10_OFEV_hydrogeo_watermass`
- `03_OFS_regbl`

In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
import ezdxf
from rasterio.plot import show
import numpy as np
from rasterio.crs import CRS
import fiona

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Hydrogeo Watermass

In [ ]:
gdb_path = f"{ROOT}data/raw/10_OFEV_hydrogeo_watermass/hydrogeo_watermass.gdb"

# List all layers in the FileGDB
layers = fiona.listlayers(gdb_path)
print("Layers:", layers)

### Main dataset

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.9.csv', index_col=0)

df_regbl.head()

## Convert current dataset into GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

In [ ]:
df_wm = gpd.read_file(gdb_path, layer='Grundwasserkoerper').to_crs('EPSG:2056')

In [ ]:
for df in [df_regbl, df_wm]:
    if 'index_right' in df.columns:
        df.drop(columns='index_right', inplace=True)


## Perform spatial joint

In [ ]:
df_regbl = gpd.sjoin(df_regbl, df_wm, how="left", predicate="within")

## Rename columns

In [ ]:
list(df_regbl.columns)

In [ ]:
cols = [
 'GWKID',
 'GWKName',
 'Flussgebiet',
 'Grundwasserleitertyp',
 'Naturraum',
 'Grundwasserregime',
 'Area',
 'SHAPE_Length',
 'SHAPE_Area',
 'Farbe',
 ]

# Add 'HydroWM -  ' to the beginning of each column name in df_regbl
df_regbl = df_regbl.rename(columns={col: f'HydroWM -  {col}' for col in cols})

In [ ]:
df_regbl.drop(columns='index_right', inplace=True)

## Convert back to DataFrame

In [ ]:
# convert back to a regular DataFrame
df_regbl = pd.DataFrame(df_regbl.drop(columns='geometry'))

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.10.csv")